# Demand-hourly surrogate — Kaggle-style walkthrough (Vibe 21)

Train / tune a **facility_kw** regressor for Unity DR knobs from an EnergyPlus DR farm.

**Honesty:** `ENERGYPLUS_SIMULATED` CANDIDATE — not BAS-validated. No future leakage in features.

**Turnkey dump:** artifacts write to `flask_app/models/` so PythonAnywhere zip + Flask reload just works.

Helpers live in `ml/` (`feature_compile_dm`, `tune_demand_hourly.SEARCH_SPACES`, `notebook_plots`, `artifact_paths`).

In [ ]:
from __future__ import annotations
import json, sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

VIBE21 = Path("..").resolve()
if not (VIBE21 / "ml").is_dir():
    VIBE21 = Path.cwd().resolve()
ML = VIBE21 / "ml"
sys.path.insert(0, str(ML))

from artifact_paths import default_model_dir, JOBLIB_NAME, CARD_NAME
from feature_compile_dm import FEATURE_COLS, assert_no_future_leakage, compile_features, matrix_xy, peak_mask
from notebook_plots import (
    dump_champion_bundle,
    extratrees_search_scatter,
    family_cv_mae_bars,
    feature_importance_bar,
    pred_vs_actual,
    residual_hist,
    run_extratrees_search,
)
from train_demand_hourly import load_farm_frame, _workspace
from tune_demand_hourly import SEARCH_SPACES, tune

assert_no_future_leakage(FEATURE_COLS)
WS = _workspace()
PQ = WS / "reports" / "dm_hourly_farm" / "dm_hourly_rows.parquet"
OUT = default_model_dir()
print("parquet:", PQ, "exists", PQ.is_file())
print("dump dir:", OUT)
print("ExtraTrees space keys:", sorted(SEARCH_SPACES["extra_trees"][1].keys()))

## 1. Load farm + feature matrix

Day-grouped rows; target `facility_kw`; 29 engineered knobs / lags (no same-hour target leakage).

In [ ]:
df = load_farm_frame(PQ)
df = compile_features(df)
X, y, groups, cols = matrix_xy(df)
peak = peak_mask(df)
print(df.shape, "days", df["day"].nunique(), "features", len(cols))
df[["day", "hour_ending", "strategy_id", "oat_c", "facility_kw"]].head()

## 2. Full family bake-off (GroupKFold)

Same `SEARCH_SPACES` as CLI `python -m ml.tune_demand_hourly`. ExtraTrees search includes
`max_features`, `min_samples_split`, `max_leaf_nodes`, `bootstrap`, wider `n_estimators`.

In [ ]:
# Moderate iters for interactive notebook; CLI defaults are higher.
result = tune(df, n_splits=5, n_iter=24, champion_refine_iter=28)
print("champion", result["champion"], result["best_params"])
print("beat persistence peak", result["beat_persistence_peak"])
pd.DataFrame(
    [{"family": e["family"], **e["oof_metrics"]} for e in result["leaderboard"]]
).sort_values("mae_peak_14_16")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
family_cv_mae_bars(result["leaderboard"], result["persistence"]["mae_peak_14_16"], ax=ax)
plt.tight_layout()
plt.show()

## 3. ExtraTrees hyperparam surface + OOF diagnostics

Validation-style scatter of CV MAE vs `n_estimators` (depth as color), then pred-vs-actual and residuals.

In [ ]:
et = run_extratrees_search(df, n_iter=48, n_splits=5)
print("ET best", et["best_params"])
print("ET OOF", et["oof_metrics"])

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
extratrees_search_scatter(et["cv_results"], ax=axes[0])
pred_vs_actual(et["y"], et["oof"], ax=axes[1])
residual_hist(et["y"], et["oof"], ax=axes[2])
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7, 5))
feature_importance_bar(et["model"], list(et["feature_cols"]), top_n=15, ax=ax)
plt.tight_layout()
plt.show()

## 4. Exploratory physics checks

OAT vs kW and one example day — sanity for DR farm fidelity.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(df["oat_c"], df["facility_kw"], s=6, alpha=0.25, c="#457b9d")
axes[0].set_xlabel("OAT °C"); axes[0].set_ylabel("facility_kw"); axes[0].set_title("OAT vs demand")
day0 = df["day"].iloc[0]
sub = df[df["day"] == day0].sort_values("hour_ending")
axes[1].plot(sub["hour_ending"], sub["facility_kw"], marker="o")
axes[1].set_xlabel("hour_ending"); axes[1].set_title(f"Example day {day0}")
plt.tight_layout()
plt.show()

## 5. Dump champion into `flask_app/models/`

Prefer full bake-off champion when it beats ExtraTrees-only; else dump ET. Flask loads this path by default.

In [ ]:
use_full = result["champion"]
full_peak = next(e["oof_metrics"]["mae_peak_14_16"] for e in result["leaderboard"] if e["family"] == use_full)
et_peak = et["oof_metrics"]["mae_peak_14_16"]
if et_peak < full_peak and use_full != "extra_trees":
    model, champ, params, mets = et["model"], "extra_trees", et["best_params"], et["oof_metrics"]
    feats = et["feature_cols"]
else:
    model, champ, params = result["model"], result["champion"], result["best_params"]
    mets = next(e["oof_metrics"] for e in result["leaderboard"] if e["family"] == champ)
    feats = result["feature_cols"]

src = "ENERGYPLUS_SIMULATED"
fs = PQ.parent / "farm_summary.json"
if fs.is_file():
    src = json.loads(fs.read_text(encoding="utf-8")).get("source", src)

art = dump_champion_bundle(
    model,
    feature_cols=list(feats),
    champion=champ,
    best_params=params,
    oof_metrics=mets,
    n_rows=result["n_rows"],
    n_days=result["n_days"],
    training_parquet=PQ,
    training_source=src,
    out_dir=OUT,
    leaderboard=result["leaderboard"],
    persistence=result["persistence"],
)
print("wrote", art)
print("card", OUT / CARD_NAME)
print("Flask will load from flask_app/models/ on next start/reload.")

## Takeaways

- GroupKFold by **day** avoids same-day leakage across hours.
- ExtraTrees with a richer space often wins peak MAE on this farm.
- Ship only to `flask_app/models/`; pack with `python tools/pack_pa_bundle.py` (≤100 MiB PA Files upload).
- Read-only HTML: Flask `GET /notebooks/demand_hourly`.